# Siamese BERT for Quora Question Similarity

This notebook builds a compact semantic duplicate detector for pairs of Quora questions. A shared BERT encoder maps both questions into the same embedding space, an element-wise L1 distance compares the embeddings, and a small classifier predicts whether the questions express the same intent.

**Experiment:** 30,000 stratified question pairs, `bert-tiny`, weighted binary cross-entropy, and three training epochs.


## Method

A Siamese network applies the same encoder to both inputs. Weight sharing ensures that both questions are represented by one consistent mapping:

$$u=f_	heta(q_1), \qquad v=f_	heta(q_2)$$

The classifier receives the element-wise L1 distance:

$$d=|u-v|, \qquad p=\sigma(g(d))$$

During training, duplicate pairs are encouraged to produce patterns of small semantic distance, while non-duplicate pairs produce larger or differently structured distances.


## 1. Environment and configuration


In [ ]:
import os
import random
import sys
from pathlib import Path
from urllib.request import urlretrieve

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.model import SiameseBERT

SEED = 42
MODEL_NAME = "prajjwal1/bert-tiny"
SAMPLE_SIZE = 30_000
MAX_LENGTH = 48
BATCH_SIZE = 32
EPOCHS = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")


## 2. Download and prepare the data


In [ ]:
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(exist_ok=True)
data_path = data_dir / "quora_duplicate_questions.tsv"

data_url = (
    "https://huggingface.co/datasets/Heliosoph/Quora-Question-Pairs/"
    "resolve/main/quora_duplicate_questions.tsv?download=true"
)

if not data_path.exists():
    print("Downloading the Quora Question Pairs dataset...")
    urlretrieve(data_url, data_path)

raw_df = pd.read_csv(data_path, sep="	")
df = (
    raw_df.dropna(subset=["question1", "question2"])
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
df["question1"] = df["question1"].astype(str)
df["question2"] = df["question2"].astype(str)

print(f"Raw rows: {len(raw_df):,}")
print(f"Clean rows: {len(df):,}")
print(df["is_duplicate"].value_counts(normalize=True).sort_index())
display(df.head())


## 3. Inspect labels and sequence lengths


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_counts = df["is_duplicate"].value_counts().sort_index()
questions = pd.concat([df["question1"], df["question2"]], ignore_index=True)
length_sample = questions.sample(20_000, random_state=SEED)
token_lengths = length_sample.map(
    lambda text: len(tokenizer.encode(text, add_special_tokens=True))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Non-duplicate", "Duplicate"], label_counts.values)
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("Question Pairs")

axes[1].hist(token_lengths[token_lengths <= token_lengths.quantile(0.99)], bins=35)
axes[1].set_title("Token Length Distribution (up to 99th percentile)")
axes[1].set_xlabel("BERT Tokens")
axes[1].set_ylabel("Questions")
plt.tight_layout()
plt.show()

print(token_lengths.describe(percentiles=[0.50, 0.90, 0.95, 0.99]))
print(f"Configured maximum length: {MAX_LENGTH}")


## 4. Create stratified train, validation, and test splits


In [ ]:
model_df = df[["question1", "question2", "is_duplicate"]].copy()

if SAMPLE_SIZE < len(model_df):
    model_df, _ = train_test_split(
        model_df,
        train_size=SAMPLE_SIZE,
        stratify=model_df["is_duplicate"],
        random_state=SEED,
    )

train_df, temporary_df = train_test_split(
    model_df,
    test_size=0.20,
    stratify=model_df["is_duplicate"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df["is_duplicate"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, split in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(split):,} rows, duplicate rate={split['is_duplicate'].mean():.4f}")


## 5. Build the PyTorch input pipeline


In [ ]:
class QuestionPairDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.question1 = frame["question1"].tolist()
        self.question2 = frame["question2"].tolist()
        self.labels = frame["is_duplicate"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def _encode(self, question):
        encoded = self.tokenizer(
            question,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return encoded["input_ids"].squeeze(0), encoded["attention_mask"].squeeze(0)

    def __getitem__(self, index):
        q1_ids, q1_mask = self._encode(self.question1[index])
        q2_ids, q2_mask = self._encode(self.question2[index])
        return {
            "q1_input_ids": q1_ids,
            "q1_attention_mask": q1_mask,
            "q2_input_ids": q2_ids,
            "q2_attention_mask": q2_mask,
            "label": torch.tensor(self.labels[index], dtype=torch.float32),
        }


train_dataset = QuestionPairDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = QuestionPairDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = QuestionPairDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
{key: tuple(value.shape) for key, value in batch.items()}


## 6. Initialize the Siamese model


In [ ]:
model = SiameseBERT(MODEL_NAME).to(device)

with torch.no_grad():
    sample_logits = model(
        batch["q1_input_ids"].to(device),
        batch["q1_attention_mask"].to(device),
        batch["q2_input_ids"].to(device),
        batch["q2_attention_mask"].to(device),
    )

print(f"Output shape: {tuple(sample_logits.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 7. Define training and evaluation utilities


In [ ]:
def run_epoch(model, data_loader, criterion, device, training=False, optimizer=None, scheduler=None):
    model.train(training)
    total_loss = 0.0
    labels_all, predictions_all = [], []

    progress = tqdm(data_loader, desc="Training" if training else "Validation")
    for batch in progress:
        labels = batch["label"].to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(
                batch["q1_input_ids"].to(device),
                batch["q1_attention_mask"].to(device),
                batch["q2_input_ids"].to(device),
                batch["q2_attention_mask"].to(device),
            )
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

        predictions = (torch.sigmoid(logits) >= 0.5).long()
        total_loss += loss.item() * labels.size(0)
        labels_all.extend(labels.long().cpu().tolist())
        predictions_all.extend(predictions.cpu().tolist())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, predictions_all, average="binary", zero_division=0
    )
    return {
        "loss": total_loss / len(labels_all),
        "accuracy": accuracy_score(labels_all, predictions_all),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


## 8. Train and save the best checkpoint


In [ ]:
negative_count = (train_df["is_duplicate"] == 0).sum()
positive_count = (train_df["is_duplicate"] == 1).sum()
pos_weight = torch.tensor(negative_count / positive_count, dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 2e-5},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.01,
)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.10),
    num_training_steps=total_steps,
)

checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)
best_model_path = checkpoint_dir / "best_siamese_bert.pt"
best_val_f1 = -1.0
history = {key: [] for key in ["train_loss", "val_loss", "train_accuracy", "val_accuracy", "train_f1", "val_f1"]}

for epoch in range(EPOCHS):
    print()
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_metrics = run_epoch(
        model, train_loader, criterion, device, True, optimizer, scheduler
    )
    val_metrics = run_epoch(model, val_loader, criterion, device)

    for metric in ["loss", "accuracy", "f1"]:
        history[f"train_{metric}"].append(train_metrics[metric])
        history[f"val_{metric}"].append(val_metrics[metric])

    print("Train:", {key: round(value, 4) for key, value in train_metrics.items()})
    print("Validation:", {key: round(value, 4) for key, value in val_metrics.items()})

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(
            {
                "model_state_dict": {
                    name: parameter.detach().cpu()
                    for name, parameter in model.state_dict().items()
                },
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "validation_f1": best_val_f1,
            },
            best_model_path,
        )

print(f"Best validation F1: {best_val_f1:.4f}")


## 9. Plot learning curves


In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for axis, metric, title in zip(
    axes,
    ["loss", "accuracy", "f1"],
    ["Loss", "Accuracy", "F1 Score"],
):
    axis.plot(epochs, history[f"train_{metric}"], marker="o", label="Training")
    axis.plot(epochs, history[f"val_{metric}"], marker="o", label="Validation")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_xticks(list(epochs))
    axis.grid(alpha=0.3)
    axis.legend()

plt.tight_layout()
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)
plt.savefig(results_dir / "training_curves.png", dpi=200, bbox_inches="tight")
plt.show()


## 10. Evaluate on the held-out test set


In [ ]:
checkpoint = torch.load(best_model_path, map_location="cpu")
model = model.to("cpu")
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

test_labels, test_probabilities = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        logits = model(
            batch["q1_input_ids"].to(device),
            batch["q1_attention_mask"].to(device),
            batch["q2_input_ids"].to(device),
            batch["q2_attention_mask"].to(device),
        )
        test_labels.extend(batch["label"].long().tolist())
        test_probabilities.extend(torch.sigmoid(logits).cpu().tolist())

test_predictions = [int(probability >= 0.5) for probability in test_probabilities]
test_metrics = {
    "accuracy": accuracy_score(test_labels, test_predictions),
    "precision": precision_score(test_labels, test_predictions),
    "recall": recall_score(test_labels, test_predictions),
    "f1": f1_score(test_labels, test_predictions),
    "roc_auc": roc_auc_score(test_labels, test_probabilities),
}

print({key: round(value, 4) for key, value in test_metrics.items()})
print(classification_report(test_labels, test_predictions, target_names=["Non-duplicate", "Duplicate"]))


In [ ]:
matrix = confusion_matrix(test_labels, test_predictions)
fpr, tpr, _ = roc_curve(test_labels, test_probabilities)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Non-duplicate", "Duplicate"],
    yticklabels=["Non-duplicate", "Duplicate"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

axes[1].plot(fpr, tpr, label=f"Siamese BERT (AUC={test_metrics['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / "evaluation_results.png", dpi=200, bbox_inches="tight")
plt.show()


## 11. Run inference on new question pairs


In [ ]:
def predict_question_pair(question1, question2, threshold=0.5):
    model.eval()
    q1 = tokenizer(
        question1,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    q2 = tokenizer(
        question2,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        logit = model(
            q1["input_ids"].to(device),
            q1["attention_mask"].to(device),
            q2["input_ids"].to(device),
            q2["attention_mask"].to(device),
        )
    probability = torch.sigmoid(logit).item()
    return {
        "duplicate_probability": probability,
        "prediction": "Duplicate" if probability >= threshold else "Non-duplicate",
    }


predict_question_pair(
    "How can I learn Python?",
    "What is the best way to study Python?",
)


## Recorded experiment

The included repository figures were produced with the configuration above.

| Metric | Test score |
|---|---:|
| Accuracy | 0.6917 |
| Precision | 0.5529 |
| Recall | 0.8591 |
| F1 | 0.6728 |
| ROC-AUC | 0.8008 |

The high recall reflects the weighted loss: the model catches most duplicate pairs while accepting more false positives.

![Training curves](../results/training_curves.png)

![Evaluation results](../results/evaluation_results.png)
